In [ ]:
# Cell 1 - Environment Setup and Configuration

import os
import pathlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import math
import time
import traceback
from datetime import datetime, timedelta
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from dotenv import load_dotenv

# Load environment variables
try:
    env_path = pathlib.Path(__file__).resolve().parent.parent / ".env"
except NameError:
    env_path = pathlib.Path(os.getcwd()) / ".env"

load_dotenv(dotenv_path=env_path)

# Verify that DATABASE_URL is loaded correctly
DATABASE_URL = os.getenv("DATABASE_URL")
print("DATABASE_URL loaded from .env:", DATABASE_URL)

# Setup SQLAlchemy engine (will use later)
from sqlalchemy import create_engine
try:
    engine = create_engine(DATABASE_URL)
    print("Database engine created successfully.")
except Exception as e:
    print(f"Error creating database engine: {e}")

# Import Supabase client
try:
    from caching.supabase_client import supabase
    print("Supabase client imported successfully.")
except Exception as e:
    print(f"Error importing Supabase client: {e}")
    
# Set paths for model storage
MODELS_DIR = pathlib.Path(os.getcwd()) / "models"
MODELS_DIR.mkdir(exist_ok=True)
PREGAME_MODEL_PATH = MODELS_DIR / "pregame_model.pkl"

print(f"Pre-game model will be saved to: {PREGAME_MODEL_PATH}")
print(f"Current date: {datetime.now().strftime('%Y-%m-%d')}")

In [1]:
# Cell 2 - Core Helper Functions

def get_team_averages(days_lookback=120):
    """
    Retrieves comprehensive team statistics from recent games.
    
    Args:
        days_lookback: Number of days to look back (default: 120, approx. 4 months)
        
    Returns:
        DataFrame with team stats including scoring, efficiency, etc.
    """
    # Calculate the date threshold
    threshold_date = (datetime.now() - timedelta(days=days_lookback)).strftime('%Y-%m-%d')
    
    try:
        # Fetch recent historical game data 
        response = supabase.table("nba_historical_game_stats")\
            .select("*")\
            .gte("game_date", threshold_date)\
            .execute()
        
        historical_data = response.data
        
        if not historical_data:
            print(f"No historical game data available from the last {days_lookback} days.")
            return pd.DataFrame()
        
        df = pd.DataFrame(historical_data)
        df['game_date'] = pd.to_datetime(df['game_date'])
        df = df.sort_values('game_date')
        
        # Calculate home team averages
        home_stats = []
        for team in df['home_team'].unique():
            team_games = df[df['home_team'] == team]
            if len(team_games) >= 3:  # At least 3 games for meaningful stats
                avg_stats = {
                    'team': team,
                    'role': 'home',
                    'num_games': len(team_games),
                    'avg_score': team_games['home_score'].mean(),
                    'avg_opp_score': team_games['away_score'].mean(),
                    'scoring_diff': team_games['home_score'].mean() - team_games['away_score'].mean(),
                    'win_pct': (team_games['home_score'] > team_games['away_score']).mean(),
                    'avg_assists': team_games['home_assists'].mean() if 'home_assists' in team_games.columns else None,
                    'avg_rebounds': team_games['home_total_reb'].mean() if 'home_total_reb' in team_games.columns else None,
                    'avg_3pm': team_games['home_3pm'].mean() if 'home_3pm' in team_games.columns else None,
                    'avg_3pa': team_games['home_3pa'].mean() if 'home_3pa' in team_games.columns else None,
                    'defensive_rating': team_games['away_score'].mean(),  # Lower is better
                    'offensive_rating': team_games['home_score'].mean(),  # Higher is better
                    'net_rating': team_games['home_score'].mean() - team_games['away_score'].mean(),
                    'recent_form': team_games.sort_values('game_date').tail(5)['home_score'].mean()
                }
                home_stats.append(avg_stats)
        
        # Calculate away team averages
        away_stats = []
        for team in df['away_team'].unique():
            team_games = df[df['away_team'] == team]
            if len(team_games) >= 3:  # At least 3 games for meaningful stats
                avg_stats = {
                    'team': team,
                    'role': 'away',
                    'num_games': len(team_games),
                    'avg_score': team_games['away_score'].mean(),
                    'avg_opp_score': team_games['home_score'].mean(),
                    'scoring_diff': team_games['away_score'].mean() - team_games['home_score'].mean(),
                    'win_pct': (team_games['away_score'] > team_games['home_score']).mean(),
                    'avg_assists': team_games['away_assists'].mean() if 'away_assists' in team_games.columns else None,
                    'avg_rebounds': team_games['away_total_reb'].mean() if 'away_total_reb' in team_games.columns else None,
                    'avg_3pm': team_games['away_3pm'].mean() if 'away_3pm' in team_games.columns else None,
                    'avg_3pa': team_games['away_3pa'].mean() if 'away_3pa' in team_games.columns else None,
                    'defensive_rating': team_games['home_score'].mean(),  # Lower is better
                    'offensive_rating': team_games['away_score'].mean(),  # Higher is better
                    'net_rating': team_games['away_score'].mean() - team_games['home_score'].mean(),
                    'recent_form': team_games.sort_values('game_date').tail(5)['away_score'].mean()
                }
                away_stats.append(avg_stats)
        
        # Combine home and away stats
        all_stats = pd.DataFrame(home_stats + away_stats)
        
        # Calculate overall team averages (combining home and away performances)
        team_overall = []
        for team in all_stats['team'].unique():
            team_data = all_stats[all_stats['team'] == team]
            
            # Calculate weighted averages based on number of games
            weights = team_data['num_games'] / team_data['num_games'].sum()
            
            overall_stats = {
                'team': team,
                'role': 'overall',
                'num_games': team_data['num_games'].sum(),
                'avg_score': np.average(team_data['avg_score'], weights=weights),
                'avg_opp_score': np.average(team_data['avg_opp_score'], weights=weights),
                'scoring_diff': np.average(team_data['scoring_diff'], weights=weights),
                'win_pct': np.average(team_data['win_pct'], weights=weights),
                'defensive_rating': np.average(team_data['defensive_rating'], weights=weights),
                'offensive_rating': np.average(team_data['offensive_rating'], weights=weights),
                'net_rating': np.average(team_data['net_rating'], weights=weights),
                'home_advantage': team_data[team_data['role'] == 'home']['avg_score'].mean() - 
                                  team_data[team_data['role'] == 'away']['avg_score'].mean() 
                                  if len(team_data) > 1 else 3.5,  # Default to 3.5 if missing
                'recent_form': np.average(team_data['recent_form'], weights=weights)
            }
            team_overall.append(overall_stats)
        
        combined_stats = pd.DataFrame(team_overall)
        print(f"Calculated overall stats for {len(combined_stats)} teams")
        
        return combined_stats
    
    except Exception as e:
        print(f"Error retrieving team averages: {e}")
        traceback.print_exc()
        return pd.DataFrame()

def get_rest_data(team, game_date):
    """
    Determines rest days for a team before a specific game
    """
    # Convert game_date to datetime if it's a string
    if isinstance(game_date, str):
        game_date = pd.to_datetime(game_date)
    
    # Look back 10 days to find the team's previous game
    lookback_date = (game_date - timedelta(days=10)).strftime('%Y-%m-%d')
    
    try:
        # Find team's previous games (as home or away)
        response_home = supabase.table("nba_historical_game_stats").select("game_date")\
            .eq("home_team", team)\
            .gte("game_date", lookback_date)\
            .lt("game_date", game_date.strftime('%Y-%m-%d'))\
            .order('game_date', desc=True)\
            .limit(1).execute()
            
        response_away = supabase.table("nba_historical_game_stats").select("game_date")\
            .eq("away_team", team)\
            .gte("game_date", lookback_date)\
            .lt("game_date", game_date.strftime('%Y-%m-%d'))\
            .order('game_date', desc=True)\
            .limit(1).execute()
        
        # Combine results to find the most recent game
        prev_games = response_home.data + response_away.data
        if not prev_games:
            # No previous game found in the lookback period
            return {'rest_days': 5, 'is_back_to_back': False}  # Assume well-rested
        
        # Sort by date, most recent first
        prev_games.sort(key=lambda x: x['game_date'], reverse=True)
        prev_game_date = pd.to_datetime(prev_games[0]['game_date'])
        
        # Calculate days between games
        rest_days = (game_date - prev_game_date).days
        is_back_to_back = (rest_days == 1)
        
        return {'rest_days': rest_days, 'is_back_to_back': is_back_to_back}
    
    except Exception as e:
        print(f"Error getting rest data for {team}: {e}")
        return {'rest_days': 2, 'is_back_to_back': False}  # Default to average rest

def get_matchup_history(home_team, away_team, max_games=5):
    """
    Gets historical matchup data between two teams
    """
    try:
        # Find games where these teams played each other
        response_home = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", home_team)\
            .eq("away_team", away_team)\
            .order('game_date', desc=True)\
            .limit(max_games).execute()
            
        response_away = supabase.table("nba_historical_game_stats").select("*")\
            .eq("home_team", away_team)\
            .eq("away_team", home_team)\
            .order('game_date', desc=True)\
            .limit(max_games).execute()
        
        # Combine results
        matchups = response_home.data + response_away.data
        
        if not matchups:
            return {
                'num_games': 0,
                'avg_point_diff': 0,  # From home team perspective
                'home_win_pct': 0.5,
                'last_winner': None,
                'avg_total_score': 0
            }
            
        # Sort by date (most recent first)
        matchups.sort(key=lambda x: x['game_date'], reverse=True)
        
        # Calculate statistics from home team perspective
        differentials = []
        home_wins = 0
        total_scores = []
        
        for game in matchups:
            if game['home_team'] == home_team:
                # Home team is our reference team
                diff = game['home_score'] - game['away_score']
                home_wins += 1 if diff > 0 else 0
                total_scores.append(game['home_score'] + game['away_score'])
            else:
                # Away team is our reference team
                diff = game['away_score'] - game['home_score']
                # No home win to count here
                total_scores.append(game['home_score'] + game['away_score'])
            differentials.append(diff)
        
        # Determine last winner
        most_recent = matchups[0]
        if most_recent['home_team'] == home_team:
            last_winner = home_team if most_recent['home_score'] > most_recent['away_score'] else away_team
        else:
            last_winner = away_team if most_recent['away_score'] > most_recent['home_score'] else home_team
        
        return {
            'num_games': len(matchups),
            'avg_point_diff': sum(differentials) / len(differentials),
            'home_win_pct': home_wins / len(response_home.data) if response_home.data else 0.5,
            'last_winner': last_winner,
            'avg_total_score': sum(total_scores) / len(total_scores) if total_scores else 0
        }
    
    except Exception as e:
        print(f"Error getting matchup history for {home_team} vs {away_team}: {e}")
        return {
            'num_games': 0,
            'avg_point_diff': 0,
            'home_win_pct': 0.5,
            'last_winner': None,
            'avg_total_score': 0
        }

def calculate_win_probability(home_stats, away_stats, matchup_data, rest_data=None):
    """
    Calculates pre-game win probability based on team stats and matchup history
    
    Args:
        home_stats: Stats dictionary for home team
        away_stats: Stats dictionary for away team  
        matchup_data: Dictionary with matchup history
        rest_data: Dictionary with rest advantage data
        
    Returns:
        Win probability for home team (0-1)
    """
    # Base win probability from net rating differential
    net_rating_diff = home_stats['net_rating'] - away_stats['net_rating']
    base_prob = 0.5 + (net_rating_diff * 0.02)  # Each point of net rating worth about 2%
    
    # Adjust for home court advantage
    home_advantage = home_stats.get('home_advantage', 3.5) * 0.015  # Each point of home advantage worth about 1.5%
    base_prob += home_advantage
    
    # Adjust for matchup history if available
    if matchup_data and matchup_data['num_games'] > 0:
        # Head-to-head advantage
        matchup_weight = min(matchup_data['num_games'] / 10, 0.5)  # Cap at 50% weight for matchup history
        matchup_advantage = matchup_data['avg_point_diff'] * 0.01 * matchup_weight
        base_prob += matchup_advantage
    
    # Adjust for rest advantage if available
    if rest_data:
        rest_advantage = rest_data['rest_advantage'] * 0.01  # Each day of rest worth about 1%
        b2b_penalty = -0.05 if rest_data['is_back_to_back_home'] else 0  # 5% penalty for home team back-to-back
        b2b_boost = 0.05 if rest_data['is_back_to_back_away'] else 0    # 5% boost if away team is on back-to-back
        
        base_prob += rest_advantage + b2b_penalty + b2b_boost
    
    # Adjust for recent form
    form_diff = (home_stats.get('recent_form', 0) - away_stats.get('recent_form', 0)) * 0.01
    base_prob += form_diff
    
    # Ensure probability is between 0 and 1
    return max(min(base_prob, 0.95), 0.05)

In [2]:
# Cell 3 - Load Raw Team Data from Database

def load_historical_games(days_lookback=365):
    """
    Loads historical game data from Supabase for feature engineering and training
    """
    # Calculate the date threshold
    threshold_date = (datetime.now() - timedelta(days=days_lookback)).strftime('%Y-%m-%d')
    print(f"Loading historical game data since {threshold_date}...")
    
    try:
        # Fetch historical game data
        response = supabase.table("nba_historical_game_stats")\
            .select("*")\
            .gte("game_date", threshold_date)\
            .order('game_date', asc=True)\
            .execute()
        
        historical_data = response.data
        
        if not historical_data:
            print(f"No historical game data available from the last {days_lookback} days.")
            return pd.DataFrame()
        
        df = pd.DataFrame(historical_data)
        df['game_date'] = pd.to_datetime(df['game_date'])
        
        print(f"Loaded {len(df)} historical games from {df['game_date'].min()} to {df['game_date'].max()}")
        print(f"Data contains {df['home_team'].nunique()} unique teams")
        
        # Display example games
        print("\nSample of recent games:")
        display(df.sort_values('game_date', ascending=False).head(5))
        
        # Check for missing data
        missing_cols = df.columns[df.isnull().mean() > 0.1]
        if not missing_cols.empty:
            print(f"\nColumns with >10% missing data: {list(missing_cols)}")
        
        return df
        
    except Exception as e:
        print(f"Error loading historical game data: {e}")
        traceback.print_exc()
        return pd.DataFrame()

# Load historical games data
historical_games = load_historical_games(days_lookback=365)

# Calculate team statistics based on this data
team_stats = get_team_averages(days_lookback=120)

# Display team stats 
if not team_stats.empty:
    print("\nTeam performance metrics (sorted by net rating):")
    display(team_stats.sort_values('net_rating', ascending=False)[
        ['team', 'avg_score', 'avg_opp_score', 'net_rating', 'win_pct', 'recent_form']
    ].head(10))
else:
    print("No team statistics available")

NameError: name 'datetime' is not defined

In [ ]:
# Cell 4 - Calculate Team-Level Metrics

def calculate_team_metrics(historical_df):
    """
    Calculates advanced team metrics for pre-game predictions 
    
    Args:
        historical_df: DataFrame with historical games
        
    Returns:
        DataFrame with team-level metrics
    """
    if historical_df.empty:
        print("No historical data provided")
        return pd.DataFrame()
    
    print("Calculating advanced team metrics...")
    
    # First calculate pace factors
    # Pace = possessions per 48 minutes
    def estimate_possessions(row):
        # Estimated possessions formula: FGA + 0.4*FTA - ORB + TOV
        # Since we don't have all these stats, we'll use a simpler estimation
        # based on the total score and typical NBA efficiency
        total_score = row['home_score'] + row['away_score']
        # NBA average: ~1.1 points per possession
        return total_score / 1.1
    
    # Add estimated possessions to each game
    historical_df['possessions'] = historical_df.apply(estimate_possessions, axis=1)
    
    # Calculate team-level statistics
    teams = set(historical_df['home_team'].unique()) | set(historical_df['away_team'].unique())
    team_metrics = []
    
    for team in teams:
        # Get home and away games
        home_games = historical_df[historical_df['home_team'] == team]
        away_games = historical_df[historical_df['away_team'] == team]
        
        # Exit if no games
        if home_games.empty and away_games.empty:
            continue
            
        total_games = len(home_games) + len(away_games)
        
        if total_games < 3:  # Need at least 3 games for meaningful stats
            continue
        
        # Calculate core metrics
        home_pts = home_games['home_score'].sum() if not home_games.empty else 0
        away_pts = away_games['away_score'].sum() if not away_games.empty else 0
        total_pts = home_pts + away_pts
        
        home_opp_pts = home_games['away_score'].sum() if not home_games.empty else 0
        away_opp_pts = away_games['home_score'].sum() if not away_games.empty else 0
        total_opp_pts = home_opp_pts + away_opp_pts
        
        # Calculate possessions
        home_poss = home_games['possessions'].sum() if not home_games.empty else 0
        away_poss = away_games['possessions'].sum() if not away_games.empty else 0
        total_poss = home_poss + away_poss
        
        # Calculate wins and win percentage
        home_wins = (home_games['home_score'] > home_games['away_score']).sum() if not home_games.empty else 0
        away_wins = (away_games['away_score'] > away_games['home_score']).sum() if not away_games.empty else 0
        total_wins = home_wins + away_wins
        
        # Calculate offensive and defensive efficiency (points per 100 possessions)
        offensive_rating = (total_pts / total_poss * 100) if total_poss > 0 else 0
        defensive_rating = (total_opp_pts / total_poss * 100) if total_poss > 0 else 0
        net_rating = offensive_rating - defensive_rating
        
        # Calculate home court advantage
        home_avg = home_games['home_score'].mean() if not home_games.empty else 0
        away_avg = away_games['away_score'].mean() if not away_games.empty else 0
        home_advantage = home_avg - away_avg if home_avg > 0 and away_avg > 0 else 3.5
        
        # Get recent form (last 10 games)
        recent_games = pd.concat([
            home_games[['game_date', 'home_score', 'away_score']].rename(
                columns={'home_score': 'team_score', 'away_score': 'opp_score'}),
            away_games[['game_date', 'away_score', 'home_score']].rename(
                columns={'away_score': 'team_score', 'home_score': 'opp_score'})
        ]).sort_values('game_date', ascending=False)
        
        recent_form = 0
        recent_opp_form = 0
        if not recent_games.empty:
            last_10 = recent_games.head(10)
            recent_form = last_10['team_score'].mean()
            recent_opp_form = last_10['opp_score'].mean()
        
        # Calculate strength of schedule
        # Get opponents' win percentages
        home_opponents = home_games['away_team'].tolist()
        away_opponents = away_games['home_team'].tolist()
        all_opponents = home_opponents + away_opponents
        
        # Add team metrics to list
        metrics = {
            'team': team,
            'games_played': total_games,
            'wins': total_wins,
            'losses': total_games - total_wins,
            'win_pct': total_wins / total_games if total_games > 0 else 0,
            'home_record': f"{home_wins}-{len(home_games) - home_wins}" if not home_games.empty else "0-0",
            'away_record': f"{away_wins}-{len(away_games) - away_wins}" if not away_games.empty else "0-0",
            'pts_per_game': total_pts / total_games if total_games > 0 else 0,
            'opp_pts_per_game': total_opp_pts / total_games if total_games > 0 else 0,
            'point_diff': (total_pts - total_opp_pts) / total_games if total_games > 0 else 0,
            'offensive_rating': offensive_rating,
            'defensive_rating': defensive_rating,
            'net_rating': net_rating,
            'home_advantage': home_advantage,
            'recent_form': recent_form,
            'recent_opp_form': recent_opp_form,
            'recent_point_diff': recent_form - recent_opp_form
        }
        
        team_metrics.append(metrics)
    
    # Convert to DataFrame
    metrics_df = pd.DataFrame(team_metrics)
    metrics_df = metrics_df.sort_values('net_rating', ascending=False)
    
    print(f"Calculated metrics for {len(metrics_df)} teams")
    
    return metrics_df

# Calculate advanced team metrics
team_metrics = calculate_team_metrics(historical_games)

# Display the top teams by net rating
if not team_metrics.empty:
    print("\nTop teams by net rating:")
    display(team_metrics[['team', 'win_pct', 'pts_per_game', 'opp_pts_per_game', 
                          'offensive_rating', 'defensive_rating', 'net_rating']].head(10))
else:
    print("No team metrics available")

In [ ]:
# Cell 5 - Enhanced Feature Engineering

def calculate_rolling_stats(historical_df, window=10):
    """
    Calculates rolling statistics for each team
    
    Args:
        historical_df: DataFrame with historical games
        window: Number of games to include in rolling window
        
    Returns:
        Dictionary mapping team names to their rolling statistics
    """
    if historical_df.empty:
        return {}
    
    print(f"Calculating {window}-game rolling statistics for each team...")
    
    teams = set(historical_df['home_team'].unique()) | set(historical_df['away_team'].unique())
    rolling_stats = {}
    
    for team in teams:
        # Get home and away games
        home_games = historical_df[historical_df['home_team'] == team].copy()
        away_games = historical_df[historical_df['away_team'] == team].copy()
        
        # Create consistent columns for team stats regardless of home/away
        home_games['team_score'] = home_games['home_score']
        home_games['opp_score'] = home_games['away_score']
        home_games['is_home'] = True
        
        away_games['team_score'] = away_games['away_score']
        away_games['opp_score'] = away_games['home_score']
        away_games['is_home'] = False
        
        # Combine and sort by date
        team_games = pd.concat([
            home_games[['game_date', 'team_score', 'opp_score', 'is_home']],
            away_games[['game_date', 'team_score', 'opp_score', 'is_home']]
        ]).sort_values('game_date')
        
        if len(team_games) < 3:  # Need at least 3 games for meaningful stats
            continue
        
        # Calculate rolling averages
        team_games['rolling_score'] = team_games['team_score'].rolling(window=window, min_periods=3).mean()
        team_games['rolling_opp_score'] = team_games['opp_score'].rolling(window=window, min_periods=3).mean()
        team_games['rolling_diff'] = team_games['rolling_score'] - team_games['rolling_opp_score']
        
        # Calculate home/away splits
        home_rolling_score = team_games[team_games['is_home']]['team_score'].rolling(window=window, min_periods=2).mean()
        away_rolling_score = team_games[~team_games['is_home']]['team_score'].rolling(window=window, min_periods=2).mean()
        
        # Get the most recent stats
        latest_stats = team_games.iloc[-1] if not team_games.empty else None
        
        if latest_stats is not None:
            rolling_stats[team] = {
                'rolling_score': latest_stats['rolling_score'],
                'rolling_opp_score': latest_stats['rolling_opp_score'],
                'rolling_diff': latest_stats['rolling_diff'],
                'home_rolling_score': home_rolling_score.iloc[-1] if not home_rolling_score.empty else None,
                'away_rolling_score': away_rolling_score.iloc[-1] if not away_rolling_score.empty else None,
                'last_10_record': (team_games.tail(10)['team_score'] > team_games.tail(10)['opp_score']).sum()
            }
    
    print(f"Calculated rolling stats for {len(rolling_stats)} teams")
    return rolling_stats

def build_pregame_features(historical_df, team_metrics_df, lookback_days=120):
    """
    Builds feature dataset for pre-game predictions
    
    Args:
        historical_df: DataFrame with historical games
        team_metrics_df: DataFrame with team metrics
        lookback_days: Days to consider for historical data
        
    Returns:
        DataFrame with features for each historical game
    """
    if historical_df.empty or team_metrics_df.empty:
        print("Missing required data for feature engineering")
        return pd.DataFrame()
    
    print("Building pre-game features dataset...")
    
    # Filter to recent games for more relevant training data
    cutoff_date = datetime.now() - timedelta(days=lookback_days)
    recent_games = historical_df[historical_df['game_date'] >= cutoff_date].copy()
    
    if recent_games.empty:
        print(f"No games found in the last {lookback_days} days")
        return pd.DataFrame()
    
    # Calculate rolling stats
    rolling_stats = calculate_rolling_stats(historical_df)
    
    # Create team metrics lookup dictionaries
    team_metrics_lookup = {row['team']: row for _, row in team_metrics_df.iterrows()}
    
    # List to store feature dictionaries
    feature_list = []
    
    # Build features for each game
    for idx, game in recent_games.iterrows():
        game_date = game['game_date']
        home_team = game['home_team']
        away_team = game['away_team']
        
        # Skip if we don't have metrics for either team
        if home_team not in team_metrics_lookup or away_team not in team_metrics_lookup:
            continue
        
        # Get team metrics
        home_metrics = team_metrics_lookup[home_team]
        away_metrics = team_metrics_lookup[away_team]
        
        # Get rolling stats if available
        home_rolling = rolling_stats.get(home_team, {})
        away_rolling = rolling_stats.get(away_team, {})
        
        # Get matchup history
        matchup = get_matchup_history(home_team, away_team)
        
        # Get rest data
        home_rest = get_rest_data(home_team, game_date)
        away_rest = get_rest_data(away_team, game_date)
        
        # Rest advantage calculation
        rest_advantage = home_rest['rest_days'] - away_rest['rest_days']
        
        # Create feature dictionary
        features = {
            # Game identifiers
            'game_id': game['game_id'],
            'game_date': game_date,
            'home_team': home_team,
            'away_team': away_team,
            
            # Team performance metrics
            'home_win_pct': home_metrics['win_pct'],
            'away_win_pct': away_metrics['win_pct'],
            'home_offensive_rating': home_metrics['offensive_rating'],
            'away_offensive_rating': away_metrics['offensive_rating'],
            'home_defensive_rating': home_metrics['defensive_rating'],
            'away_defensive_rating': away_metrics['defensive_rating'],
            'home_net_rating': home_metrics['net_rating'],
            'away_net_rating': away_metrics['net_rating'],
            'net_rating_diff': home_metrics['net_rating'] - away_metrics['net_rating'],
            
            # Rolling statistics
            'home_rolling_score': home_rolling.get('rolling_score'),
            'away_rolling_score': away_rolling.get('rolling_score'),
            'home_rolling_opp_score': home_rolling.get('rolling_opp_score'),
            'away_rolling_opp_score': away_rolling.get('rolling_opp_score'),
            'rolling_score_diff': home_rolling.get('rolling_score', 0) - away_rolling.get('rolling_score', 0),
            
            # Recent form
            'home_recent_form': home_metrics['recent_form'],
            'away_recent_form': away_metrics['recent_form'],
            'home_last_10_record': home_rolling.get('last_10_record'),
            'away_last_10_record': away_rolling.get('last_10_record'),
            
            # Rest factors
            'home_rest_days': home_rest['rest_days'],
            'away_rest_days': away_rest['rest_days'],
            'rest_advantage': rest_advantage,
            'home_back_to_back': int(home_rest['is_back_to_back']),
            'away_back_to_back': int(away_rest['is_back_to_back']),
            
            # Matchup history
            'matchup_games': matchup['num_games'],
            'matchup_point_diff': matchup['avg_point_diff'],
            'home_historical_win_pct': matchup['home_win_pct'],
            
            # Home court advantage
            'home_advantage': home_metrics['home_advantage'],
            
            # Target variables
            'home_score': game['home_score'],
            'away_score': game['away_score'],
            'home_win': 1 if game['home_score'] > game['away_score'] else 0,
            'point_diff': game['home_score'] - game['away_score'],
            'total_score': game['home_score'] + game['away_score']
        }
        
        feature_list.append(features)
    
    # Convert to DataFrame
    features_df = pd.DataFrame(feature_list)
    
    # Fill NaN values with reasonable defaults
    numeric_cols = features_df.select_dtypes(include=np.number).columns
    for col in numeric_cols:
        if features_df[col].isnull().any():
            if 'rating' in col or 'score' in col:
                # Use median for ratings and scores
                features_df[col] = features_df[col].fillna(features_df[col].median())
            elif 'rest' in col:
                # Default rest to 2 days
                features_df[col] = features_df[col].fillna(2)
            elif 'back_to_back' in col:
                # Default to 0 (not a back-to-back)
                features_df[col] = features_df[col].fillna(0)
            else:
                # Use 0 for other numerics
                features_df[col] = features_df[col].fillna(0)
    
    print(f"Created features for {len(features_df)} games")
    
    return features_df

# Build pre-game features dataset
pregame_features = build_pregame_features(historical_games, team_metrics)

# Display the features
if not pregame_features.empty:
    print("\nSample of pre-game features:")
    display(pregame_features.head())
    
    # Show feature statistics
    print("\nFeature statistics:")
    display(pregame_features.describe())
    
    # Show correlation with target variables
    print("\nCorrelation with home score:")
    corr_with_home_score = pregame_features.corr()['home_score'].sort_values(ascending=False)
    display(corr_with_home_score.head(10))
    display(corr_with_home_score.tail(10))
    
    print("\nCorrelation with point differential (home_score - away_score):")
    corr_with_diff = pregame_features.corr()['point_diff'].sort_values(ascending=False)
    display(corr_with_diff.head(10))
    
    # Save features for later use
    features_file = MODELS_DIR / "pregame_features.csv"
    pregame_features.to_csv(features_file, index=False)
    print(f"Saved pre-game features to {features_file}")
else:
    print("No features available for analysis")

In [ ]:
# Cell 6 - Train and Evaluate Pre-Game Prediction Model

def train_pregame_model(features_df, target='home_score'):
    """
    Trains a model for pre-game predictions
    
    Args:
        features_df: DataFrame with pre-game features
        target: Target variable to predict ('home_score', 'away_score', 'point_diff', or 'total_score')
        
    Returns:
        Trained model and validation metrics
    """
    if features_df.empty:
        print("No training data available")
        return None, {}
    
    print(f"Training pre-game prediction model for target: {target}")
    
    # Define feature columns (excluding game identifiers and target variables)
    feature_columns = [
        'home_win_pct', 'away_win_pct',
        'home_offensive_rating', 'away_offensive_rating',
        'home_defensive_rating', 'away_defensive_rating',
        'home_net_rating', 'away_net_rating', 'net_rating_diff',
        'home_rolling_score', 'away_rolling_score',
        'home_rolling_opp_score', 'away_rolling_opp_score',
        'rolling_score_diff',
        'home_recent_form', 'away_recent_form',
        'home_last_10_record', 'away_last_10_record',
        'home_rest_days', 'away_rest_days', 'rest_advantage',
        'home_back_to_back', 'away_back_to_back',
        'matchup_games', 'matchup_point_diff', 'home_historical_win_pct',
        'home_advantage'
    ]
    
    # Ensure all feature columns exist
    missing_cols = [col for col in feature_columns if col not in features_df.columns]
    if missing_cols:
        print(f"Warning: Missing feature columns: {missing_cols}")
        feature_columns = [col for col in feature_columns if col in features_df.columns]
    
    # Prepare feature matrix and target vector
    X = features_df[feature_columns]
    y = features_df[target]
    
    # Print feature info
    print(f"Training with {len(X)} samples and {len(feature_columns)} features")
    
    # Sort by date to ensure time-based validation
    if 'game_date' in features_df.columns:
        sorted_indices = features_df['game_date'].sort_values().index
        X = X.loc[sorted_indices]
        y = y.loc[sorted_indices]
    
    # Split data into training and testing sets (using time-based split)
    train_size = int(0.8 * len(X))
    X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
    y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]
    
    # Define model
    model = GradientBoostingRegressor(
        n_estimators=200, 
        learning_rate=0.1,
        max_depth=4,
        random_state=42,
        subsample=0.8
    )
    
    # Train model
    model.fit(X_train, y_train)
    
    # Evaluate on training and test sets
    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)
    
    # Calculate metrics
    train_mse = mean_squared_error(y_train, train_preds)
    test_mse = mean_squared_error(y_test, test_preds)
    train_mae = mean_absolute_error(y_train, train_preds)
    test_mae = mean_absolute_error(y_test, test_preds)
    r2 = r2_score(y_test, test_preds)
    
    metrics = {
        'train_mse': train_mse,
        'test_mse': test_mse,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'r2': r2,
        'feature_importance': dict(zip(feature_columns, model.feature_importances_))
    }
    
    print(f"Training MSE: {train_mse:.2f}, MAE: {train_mae:.2f}")
    print(f"Test MSE: {test_mse:.2f}, MAE: {test_mae:.2f}")
    print(f"R² Score: {r2:.4f}")
    
    # Save model
    joblib.dump(model, PREGAME_MODEL_PATH)
    print(f"Model saved to {PREGAME_MODEL_PATH}")
    
    return model, metrics

def train_multiple_models(features_df):
    """
    Trains multiple models for different prediction targets
    
    Args:
        features_df: DataFrame with pre-game features
        
    Returns:
        Dictionary of trained models and metrics
    """
    models = {}
    
    # Train home score model
    print("\n=== TRAINING HOME SCORE MODEL ===")
    home_model, home_metrics = train_pregame_model(features_df, target='home_score')
    models['home_score'] = {
        'model': home_model,
        'metrics': home_metrics
    }
    
    # Train away score model
    print("\n=== TRAINING AWAY SCORE MODEL ===")
    away_model, away_metrics = train_pregame_model(features_df, target='away_score')
    models['away_score'] = {
        'model': away_model,
        'metrics': away_metrics
    }
    
    # Train point differential model
    print("\n=== TRAINING POINT DIFFERENTIAL MODEL ===")
    diff_model, diff_metrics = train_pregame_model(features_df, target='point_diff')
    models['point_diff'] = {
        'model': diff_model,
        'metrics': diff_metrics
    }
    
    # Train total score model
    print("\n=== TRAINING TOTAL SCORE MODEL ===")
    total_model, total_metrics = train_pregame_model(features_df, target='total_score')
    models['total_score'] = {
        'model': total_model,
        'metrics': total_metrics
    }
    
    return models

# Train models if we have features
if not pregame_features.empty:
    models = train_multiple_models(pregame_features)
    
    # Set home score model as the default pregame model
    pregame_model = models['home_score']['model']
    
    # Save all models to files
    for target, model_info in models.items():
        model_path = MODELS_DIR / f"pregame_{target}_model.pkl"
        joblib.dump(model_info['model'], model_path)
        print(f"Saved {target} model to {model_path}")
else:
    print("Cannot train models - no features available")

In [ ]:
# Cell 7 - Generate Predictions for Upcoming Games

def get_upcoming_games(days=7):
    """
    Fetches upcoming NBA games scheduled in the next few days
    
    Args:
        days: Number of days to look ahead
        
    Returns:
        DataFrame with upcoming games
    """
    today = datetime.now()
    end_date = (today + timedelta(days=days)).strftime('%Y-%m-%d')
    today_str = today.strftime('%Y-%m-%d')
    
    print(f"Looking for upcoming games from {today_str} to {end_date}...")
    
    try:
        # Try first from nba_game_schedule table
        response = supabase.table("nba_game_schedule")\
            .select("*")\
            .gte("game_date", today_str)\
            .lte("game_date", end_date)\
            .execute()
        
        if response.data:
            df = pd.DataFrame(response.data)
            print(f"Found {len(df)} upcoming games from schedule table")
            return df
            
        # If no scheduled games found, use previous approach to predict common matchups
        print("No scheduled games found, generating sample matchups...")
        
        # Get all teams that have played games recently
        past_threshold = (today - timedelta(days=30)).strftime('%Y-%m-%d')
        response = supabase.table("nba_historical_game_stats")\
            .select("home_team,away_team")\
            .gte("game_date", past_threshold)\
            .execute()
        
        if not response.data:
            print("No recent games found to generate matchups")
            return pd.DataFrame()
            
        recent_games = pd.DataFrame(response.data)
        
        # Get unique teams
        teams = set(recent_games['home_team'].unique()) | set(recent_games['away_team'].unique())
        teams = list(teams)
        
        if len(teams) < 2:
            print("Not enough teams found to generate matchups")
            return pd.DataFrame()
        
        # Generate some sample matchups for upcoming days
        upcoming_games = []
        for i in range(min(days * 5, 20)):  # Generate up to 20 games
            # Select random home and away teams
            import random
            home_idx = random.randint(0, len(teams) - 1)
            away_idx = random.randint(0, len(teams) - 1)
            while away_idx == home_idx:  # Ensure teams are different
                away_idx = random.randint(0, len(teams) - 1)
                
            home_team = teams[home_idx]
            away_team = teams[away_idx]
            
            # Generate future game date
            game_date = (today + timedelta(days=random.randint(0, days))).strftime('%Y-%m-%d')
            
            upcoming_games.append({
                'game_id': 1000000 + i,  # Dummy ID
                'game_date': game_date,
                'home_team': home_team,
                'away_team': away_team,
                'is_generated': True  # Flag to indicate this is a generated matchup
            })
        
        df = pd.DataFrame(upcoming_games)
        print(f"Generated {len(df)} sample matchups")
        return df
        
    except Exception as e:
        print(f"Error fetching upcoming games: {e}")
        traceback.print_exc()
        return pd.DataFrame()

def predict_upcoming_game(game, team_metrics, models):
    """
    Makes predictions for an upcoming game using pre-trained models
    
    Args:
        game: Dictionary or Series with game information (home_team, away_team, game_date)
        team_metrics: DataFrame with team metrics
        models: Dictionary of trained models for different targets
        
    Returns:
        Dictionary with predictions and game information
    """
    home_team = game['home_team']
    away_team = game['away_team']
    game_date = pd.to_datetime(game['game_date'])
    
    # Check if we have metrics for both teams
    team_metrics_dict = {row['team']: row for _, row in team_metrics.iterrows()}
    if home_team not in team_metrics_dict or away_team not in team_metrics_dict:
        print(f"Missing team metrics for {home_team} or {away_team}")
        return None
    
    # Get team metrics
    home_metrics = team_metrics_dict[home_team]
    away_metrics = team_metrics_dict[away_team]
    
    # Get matchup history
    matchup = get_matchup_history(home_team, away_team)
    
    # Get rest data
    home_rest = get_rest_data(home_team, game_date)
    away_rest = get_rest_data(away_team, game_date)
    
    # Create feature dictionary (similar to build_pregame_features)
    features = {
        # Game identifiers
        'game_id': game['game_id'] if 'game_id' in game else 0,
        'game_date': game_date,
        'home_team': home_team,
        'away_team': away_team,
        
        # Team performance metrics
        'home_win_pct': home_metrics['win_pct'],
        'away_win_pct': away_metrics['win_pct'],
        'home_offensive_rating': home_metrics['offensive_rating'],
        'away_offensive_rating': away_metrics['offensive_rating'],
        'home_defensive_rating': home_metrics['defensive_rating'],
        'away_defensive_rating': away_metrics['defensive_rating'],
        'home_net_rating': home_metrics['net_rating'],
        'away_net_rating': away_metrics['net_rating'],
        'net_rating_diff': home_metrics['net_rating'] - away_metrics['net_rating'],
        
        # We may not have rolling stats for simulation
        'home_rolling_score': home_metrics.get('recent_form', home_metrics['pts_per_game']),
        'away_rolling_score': away_metrics.get('recent_form', away_metrics['pts_per_game']),
        'home_rolling_opp_score': home_metrics.get('recent_opp_form', home_metrics['opp_pts_per_game']),
        'away_rolling_opp_score': away_metrics.get('recent_opp_form', away_metrics['opp_pts_per_game']),
        'rolling_score_diff': home_metrics.get('recent_form', home_metrics['pts_per_game']) - 
                              away_metrics.get('recent_form', away_metrics['pts_per_game']),
        
        # Recent form
        'home_recent_form': home_metrics['recent_form'],
        'away_recent_form': away_metrics['recent_form'],
        'home_last_10_record': 5,  # Default to 5-5 record if not available
        'away_last_10_record': 5,
        
        # Rest factors
        'home_rest_days': home_rest['rest_days'],
        'away_rest_days': away_rest['rest_days'],
        'rest_advantage': home_rest['rest_days'] - away_rest['rest_days'],
        'home_back_to_back': int(home_rest['is_back_to_back']),
        'away_back_to_back': int(away_rest['is_back_to_back']),
        
        # Matchup history
        'matchup_games': matchup['num_games'],
        'matchup_point_diff': matchup['avg_point_diff'],
        'home_historical_win_pct': matchup['home_win_pct'],
        
        # Home court advantage
        'home_advantage': home_metrics['home_advantage']
    }
    
    # Convert to DataFrame for prediction
    X = pd.DataFrame([features])
    
    # Make predictions using models
    predictions = {}
    for target, model_info in models.items():
        model = model_info['model']
        
        # Get required features for this model
        if hasattr(model, 'feature_names_in_'):
            model_features = [f for f in model.feature_names_in_ if f in X.columns]
        else:
            # Default set of features if not available
            model_features = [f for f in X.columns if f not in ['game_id', 'game_date', 'home_team', 'away_team']]
        
        # Make prediction
        X_model = X[model_features]
        predictions[target] = model.predict(X_model)[0]
    
    # Calculate win probability
    win_prob = calculate_win_probability(home_metrics, away_metrics, matchup, {
        'rest_advantage': features['rest_advantage'],
        'is_back_to_back_home': features['home_back_to_back'],
        'is_back_to_back_away': features['away_back_to_back']
    })
    
    # Ensure consistent predictions
    # If we have separate home and away models, use those
    if 'home_score' in predictions and 'away_score' in predictions:
        home_score = predictions['home_score']
        away_score = predictions['away_score']
        point_diff = home_score - away_score
    
    # If we have point diff model but not separate scores
    elif 'point_diff' in predictions and ('home_score' not in predictions or 'away_score' not in predictions):
        point_diff = predictions['point_diff']
        
        # Estimate scores based on team averages and point diff
        home_avg = home_metrics['pts_per_game'] 
        away_avg = away_metrics['pts_per_game']
        
        # Adjust by half the point diff each way
        home_score = home_avg + (point_diff / 2)
        away_score = away_avg - (point_diff / 2)
    
    # Default to team averages if no prediction models
    else:
        home_score = home_metrics['pts_per_game']
        away_score = away_metrics['pts_per_game']
        point_diff = home_score - away_score
    
    # If we have a total score model, ensure consistency
    if 'total_score' in predictions:
        total_predicted = predictions['total_score']
        
        # Adjust individual scores to match total while maintaining point diff
        current_total = home_score + away_score
        if current_total > 0:  # Avoid division by zero
            adjustment_ratio = total_predicted / current_total
            
            # Apply adjustment
            home_score = home_score * adjustment_ratio
            away_score = away_score * adjustment_ratio
        else:
            # Fallback to simple split of total
            home_score = total_predicted * (0.5 + (point_diff / 200))  # Slight adjustment based on diff
            away_score = total_predicted - home_score
    
    # Round scores to 1 decimal place
    home_score = round(home_score, 1)
    away_score = round(away_score, 1)
    total_score = home_score + away_score
    
    # Compile results
    result = {
        'game_id': features['game_id'],
        'game_date': features['game_date'],
        'home_team': home_team,
        'away_team': away_team,
        'predicted_home_score': home_score,
        'predicted_away_score': away_score,
        'predicted_point_diff': home_score - away_score,
        'predicted_total_score': total_score,
        'win_probability': win_prob,
        'home_rest': features['home_rest_days'],
        'away_rest': features['away_rest_days'],
        'home_back_to_back': bool(features['home_back_to_back']),
        'away_back_to_back': bool(features['away_back_to_back']),
        'historical_matchups': features['matchup_games'],
        'prediction_timestamp': datetime.now(),
        'models_used': list(models.keys())
    }
    
    return result

def predict_all_upcoming_games(models):
    """
    Predicts outcomes for all upcoming games
    
    Args:
        models: Dictionary of trained models
        
    Returns:
        DataFrame with predictions for upcoming games
    """
    # First check if models exist
    if not models or not any(m['model'] is not None for m in models.values()):
        print("No trained models available for prediction")
        return pd.DataFrame()
    
    # Get upcoming games
    upcoming_games = get_upcoming_games(days=7)
    if upcoming_games.empty:
        print("No upcoming games found to predict")
        return pd.DataFrame()
    
    print(f"Making predictions for {len(upcoming_games)} upcoming games...")
    
    # Get latest team metrics
    team_metrics = calculate_team_metrics(historical_games)
    if team_metrics.empty:
        print("No team metrics available")
        return pd.DataFrame()
    
    # Make predictions for each game
    predictions = []
    for _, game in upcoming_games.iterrows():
        prediction = predict_upcoming_game(game, team_metrics, models)
        if prediction:
            predictions.append(prediction)
            print(f"Predicted {prediction['home_team']} vs {prediction['away_team']}: " +
                  f"{prediction['predicted_home_score']}-{prediction['predicted_away_score']} " +
                  f"(Win prob: {prediction['win_probability']:.1%})")
    
    # Convert to DataFrame
    if predictions:
        predictions_df = pd.DataFrame(predictions)
        
        # Sort by date
        if 'game_date' in predictions_df.columns:
            predictions_df = predictions_df.sort_values('game_date')
        
        return predictions_df
    else:
        print("No predictions generated")
        return pd.DataFrame()

# Try to load existing models
try:
    # Try to load saved models
    saved_models = {}
    
    # Home score model
    home_model_path = MODELS_DIR / "pregame_home_score_model.pkl"
    if home_model_path.exists():
        home_model = joblib.load(home_model_path)
        saved_models['home_score'] = {'model': home_model, 'metrics': {}}
        print("Loaded home score model")
    
    # Away score model
    away_model_path = MODELS_DIR / "pregame_away_score_model.pkl"
    if away_model_path.exists():
        away_model = joblib.load(away_model_path)
        saved_models['away_score'] = {'model': away_model, 'metrics': {}}
        print("Loaded away score model")
    
    # Point diff model
    diff_model_path = MODELS_DIR / "pregame_point_diff_model.pkl"
    if diff_model_path.exists():
        diff_model = joblib.load(diff_model_path)
        saved_models['point_diff'] = {'model': diff_model, 'metrics': {}}
        print("Loaded point differential model")
    
    # Total score model
    total_model_path = MODELS_DIR / "pregame_total_score_model.pkl"
    if total_model_path.exists():
        total_model = joblib.load(total_model_path)
        saved_models['total_score'] = {'model': total_model, 'metrics': {}}
        print("Loaded total score model")
    
    # Use models from training if available, otherwise use loaded models
    prediction_models = models if 'models' in globals() and models else saved_models
    
    # Make predictions if we have at least one model
    if prediction_models:
        upcoming_predictions = predict_all_upcoming_games(prediction_models)
        
        if not upcoming_predictions.empty:
            print("\nAll upcoming game predictions:")
            display(upcoming_predictions[['game_date', 'home_team', 'away_team', 
                                         'predicted_home_score', 'predicted_away_score',
                                         'win_probability']])
    else:
        print("No models available for prediction")
except Exception as e:
    print(f"Error making predictions: {e}")
    traceback.print_exc()

In [ ]:
# Cell 8 - Visualize Feature Importance and Evaluation

def visualize_feature_importance(models):
    """
    Creates visualizations of feature importance for the trained models
    
    Args:
        models: Dictionary of trained models and metrics
    """
    if not models:
        print("No models available for visualization")
        return
    
    # Create figure for feature importance
    plt.figure(figsize=(14, 10))
    
    for i, (target, model_info) in enumerate(models.items()):
        if 'metrics' not in model_info or 'feature_importance' not in model_info['metrics']:
            continue
            
        # Get feature importance
        importance = model_info['metrics']['feature_importance']
        
        # Convert to DataFrame and sort
        importance_df = pd.DataFrame({
            'Feature': list(importance.keys()),
            'Importance': list(importance.values())
        }).sort_values('Importance', ascending=False)
        
        # Plot on subplot
        plt.subplot(2, 2, i+1)
        sns.barplot(x='Importance', y='Feature', data=importance_df.head(10))
        plt.title(f'Top 10 Features for {target.replace("_", " ").title()} Model')
        plt.tight_layout()
    
    plt.suptitle('Feature Importance by Model', y=1.02, fontsize=16)
    plt.tight_layout()
    plt.show()
    
    # Feature group importance
    plt.figure(figsize=(14, 10))
    
    for i, (target, model_info) in enumerate(models.items()):
        if 'metrics' not in model_info or 'feature_importance' not in model_info['metrics']:
            continue
            
        # Get feature importance
        importance = model_info['metrics']['feature_importance']
        
        # Group features by category
        feature_groups = {
            'Team Performance': [f for f in importance.keys() if 'win_pct' in f or 'rating' in f],
            'Recent Form': [f for f in importance.keys() if 'recent_form' in f or 'rolling' in f or 'last_10' in f],
            'Rest Factors': [f for f in importance.keys() if 'rest' in f or 'back_to_back' in f],
            'Matchup History': [f for f in importance.keys() if 'matchup' in f or 'historical' in f],
            'Home Advantage': [f for f in importance.keys() if 'home_advantage' in f]
        }
        
        # Calculate importance by group
        group_importance = {}
        for group, features in feature_groups.items():
            group_importance[group] = sum(importance[f] for f in features if f in importance)
        
        # Plot on subplot
        plt.subplot(2, 2, i+1)
        plt.pie(
            group_importance.values(), 
            labels=group_importance.keys(),
            autopct='%1.1f%%',
            explode=[0.05] * len(group_importance),
            shadow=True
        )
        plt.title(f'Feature Group Importance for {target.replace("_", " ").title()} Model')
    
    plt.suptitle('Feature Group Importance by Model', y=1.02, fontsize=16)
    plt.tight_layout()
    plt.show()

def visualize_prediction_performance(models, features_df):
    """
    Visualizes prediction performance on test data
    
    Args:
        models: Dictionary of trained models
        features_df: DataFrame with features and actual outcomes
    """
    if not models or features_df.empty:
        print("No models or feature data available for visualization")
        return
    
    # Create figure for prediction vs actual
    plt.figure(figsize=(14, 10))
    
    # Sort by date
    if 'game_date' in features_df.columns:
        sorted_df = features_df.sort_values('game_date')
    else:
        sorted_df = features_df
    
    # Use 20% of data as test set
    test_size = int(0.2 * len(sorted_df))
    test_df = sorted_df.iloc[-test_size:]
    
    for i, (target, model_info) in enumerate(models.items()):
        if 'model' not in model_info or model_info['model'] is None:
            continue
            
        # Get model
        model = model_info['model']
        
        # Get feature columns
        if hasattr(model, 'feature_names_in_'):
            feature_cols = model.feature_names_in_
        else:
            # Default set of features
            feature_cols = [c for c in features_df.columns if c not in 
                           ['game_id', 'game_date', 'home_team', 'away_team', 
                            'home_score', 'away_score', 'point_diff', 'total_score', 'home_win']]
            
        # Make predictions on test data
        X_test = test_df[feature_cols]
        y_test = test_df[target]
        predictions = model.predict(X_test)
        
        # Plot on subplot
        plt.subplot(2, 2, i+1)
        
        # Create comparison DataFrame
        results = pd.DataFrame({
            'Actual': y_test.values,
            'Predicted': predictions,
            'Game': test_df['home_team'] + ' vs ' + test_df['away_team'],
            'Date': pd.to_datetime(test_df['game_date']) if 'game_date' in test_df.columns else range(len(y_test))
        })
        
        # Plot
        plt.scatter(results['Actual'], results['Predicted'], alpha=0.6)
        
        # Add reference line
        min_val = min(results['Actual'].min(), results['Predicted'].min())
        max_val = max(results['Actual'].max(), results['Predicted'].max())
        plt.plot([min_val, max_val], [min_val, max_val], 'r--')
        
        # Annotations
        plt.xlabel('Actual')
        plt.ylabel('Predicted')
        plt.title(f'{target.replace("_", " ").title()} Predictions vs Actual')
        
        # Calculate metrics
        mse = mean_squared_error(results['Actual'], results['Predicted'])
        mae = mean_absolute_error(results['Actual'], results['Predicted'])
        r2 = r2_score(results['Actual'], results['Predicted'])
        
        # Add metrics to plot
        plt.annotate(f'MSE: {mse:.2f}\nMAE: {mae:.2f}\nR²: {r2:.4f}',
                    xy=(0.05, 0.95), xycoords='axes fraction',
                    ha='left', va='top',
                    bbox=dict(boxstyle='round', fc='white', alpha=0.8))
    
    plt.suptitle('Prediction Performance by Model', y=1.02, fontsize=16)
    plt.tight_layout()
    plt.show()
    
    # Point differential visualization
    if 'point_diff' in models and 'model' in models['point_diff'] and models['point_diff']['model'] is not None:
        model = models['point_diff']['model']
        
        # Get feature columns
        if hasattr(model, 'feature_names_in_'):
            feature_cols = model.feature_names_in_
        else:
            # Default set of features
            feature_cols = [c for c in features_df.columns if c not in 
                           ['game_id', 'game_date', 'home_team', 'away_team', 
                            'home_score', 'away_score', 'point_diff', 'total_score', 'home_win']]
            
        # Make predictions on test data
        X_test = test_df[feature_cols]
        y_test = test_df['point_diff']
        predictions = model.predict(X_test)
        
        # Create comparison DataFrame
        results = pd.DataFrame({
            'Actual': y_test.values,
            'Predicted': predictions,
            'Outcome': ['Home Win' if d > 0 else 'Away Win' if d < 0 else 'Tie' for d in y_test.values],
            'Predicted Outcome': ['Home Win' if d > 0 else 'Away Win' if d < 0 else 'Tie' for d in predictions],
            'Correct': [1 if (a > 0 and p > 0) or (a < 0 and p < 0) or (a == 0 and p == 0) else 0 
                       for a, p in zip(y_test.values, predictions)],
            'Game': test_df['home_team'] + ' vs ' + test_df['away_team'],
            'Date': pd.to_datetime(test_df['game_date']) if 'game_date' in test_df.columns else range(len(y_test))
        })
        
        # Calculate win prediction accuracy
        win_accuracy = results['Correct'].mean() * 100
        
        # Plot point differential
        plt.figure(figsize=(12, 8))
        
        # Plot actual vs predicted
        plt.subplot(2, 1, 1)
        plt.scatter(results['Actual'], results['Predicted'], 
                   c=['g' if c == 1 else 'r' for c in results['Correct']], alpha=0.6)
        
        # Add reference line
        min_val = min(results['Actual'].min(), results['Predicted'].min())
        max_val = max(results['Actual'].max(), results['Predicted'].max())
        plt.plot([min_val, max_val], [min_val, max_val], 'k--')
        plt.plot([0, 0], [min_val, max_val], 'b-', alpha=0.3)  # Vertical line at x=0
        plt.plot([min_val, max_val], [0, 0], 'b-', alpha=0.3)  # Horizontal line at y=0
        
        # Annotations
        plt.xlabel('Actual Point Differential')
        plt.ylabel('Predicted Point Differential')
        plt.title('Point Differential Predictions')
        
        # Add metrics to plot
        mse = mean_squared_error(results['Actual'], results['Predicted'])
        mae = mean_absolute_error(results['Actual'], results['Predicted'])
        
        plt.annotate(f'MSE: {mse:.2f}\nMAE: {mae:.2f}\nWin Prediction Accuracy: {win_accuracy:.1f}%',
                    xy=(0.05, 0.95), xycoords='axes fraction',
                    ha='left', va='top',
                    bbox=dict(boxstyle='round', fc='white', alpha=0.8))
        
        # Plot win prediction confusion matrix
        plt.subplot(2, 1, 2)
        
        # Create confusion matrix
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(
            results['Outcome'].map({'Home Win': 1, 'Away Win': 0, 'Tie': 2}),
            results['Predicted Outcome'].map({'Home Win': 1, 'Away Win': 0, 'Tie': 2})
        )
        
        # Plot heatmap
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=['Away Win', 'Home Win', 'Tie'] if cm.shape[1] >= 3 else ['Away Win', 'Home Win'],
                   yticklabels=['Away Win', 'Home Win', 'Tie'] if cm.shape[0] >= 3 else ['Away Win', 'Home Win'])
        plt.title('Win Prediction Confusion Matrix')
        plt.xlabel('Predicted Outcome')
        plt.ylabel('Actual Outcome')
        
        plt.tight_layout()
        plt.show()

# Check if we have models and features for visualization
if 'models' in globals() and models and not pregame_features.empty:
    print("Visualizing model performance and feature importance...")
    visualize_feature_importance(models)
    visualize_prediction_performance(models, pregame_features)
else:
    print("No models or features available for visualization")

In [ ]:
# Cell 9 - Win Probability and Game Simulation

def simulate_game(home_team, away_team, team_metrics, models=None, num_simulations=1000):
    """
    Simulates a game multiple times to generate win probability distribution
    
    Args:
        home_team: Name of home team
        away_team: Name of away team
        team_metrics: DataFrame with team metrics
        models: Dictionary of trained models (optional)
        num_simulations: Number of simulations to run
        
    Returns:
        Dictionary with simulation results
    """
    # Convert team names to DataFrame format
    if isinstance(team_metrics, pd.DataFrame):
        team_metrics_dict = {row['team']: row for _, row in team_metrics.iterrows()}
    else:
        team_metrics_dict = team_metrics
    
    # Check if we have metrics for both teams
    if home_team not in team_metrics_dict or away_team not in team_metrics_dict:
        print(f"Missing team metrics for {home_team} or {away_team}")
        return None
    
    # Get team metrics
    home_metrics = team_metrics_dict[home_team]
    away_metrics = team_metrics_dict[away_team]
    
    # Use today's date for game date
    game_date = datetime.now()
    
    # Get matchup history
    matchup = get_matchup_history(home_team, away_team)
    
    # Get rest data
    home_rest = get_rest_data(home_team, game_date)
    away_rest = get_rest_data(away_team, game_date)
    
    # Calculate base win probability
    base_win_prob = calculate_win_probability(home_metrics, away_metrics, matchup, {
        'rest_advantage': home_rest['rest_days'] - away_rest['rest_days'],
        'is_back_to_back_home': home_rest['is_back_to_back'],
        'is_back_to_back_away': away_rest['is_back_to_back']
    })
    
    # Create game dictionary
    game = {
        'game_id': 999999,  # Placeholder
        'game_date': game_date,
        'home_team': home_team,
        'away_team': away_team
    }
    
    # Get prediction if models available
    base_prediction = None
    if models:
        base_prediction = predict_upcoming_game(game, team_metrics_dict, models)
    
    # Run simulations
    print(f"Running {num_simulations} simulations for {home_team} vs {away_team}...")
    
    home_wins = 0
    home_scores = []
    away_scores = []
    point_diffs = []
    total_scores = []
    
    # Determine simulation parameters
    if base_prediction:
        # Use model predictions as base
        home_mean = base_prediction['predicted_home_score']
        away_mean = base_prediction['predicted_away_score']
        
        # Estimate standard deviation (typical NBA variation is around 6-8 points)
        home_std = 7.0
        away_std = 7.0
    else:
        # Use team averages as base
        home_mean = home_metrics['pts_per_game']
        away_mean = away_metrics['pts_per_game']
        
        # Higher uncertainty without models
        home_std = 8.0
        away_std = 8.0
    
    # Run simulations
    import numpy as np
    
    for _ in range(num_simulations):
        # Simulate scores using normal distribution
        home_score = max(0, np.random.normal(home_mean, home_std))
        away_score = max(0, np.random.normal(away_mean, away_std))
        
        # Record results
        home_scores.append(home_score)
        away_scores.append(away_score)
        point_diffs.append(home_score - away_score)
        total_scores.append(home_score + away_score)
        
        # Count wins
        if home_score > away_score:
            home_wins += 1
    
    # Calculate win probability
    simulated_win_prob = home_wins / num_simulations
    
    # Calculate statistics
    results = {
        'home_team': home_team,
        'away_team': away_team,
        'base_win_probability': base_win_prob,
        'simulated_win_probability': simulated_win_prob,
        'expected_home_score': np.mean(home_scores),
        'expected_away_score': np.mean(away_scores),
        'expected_point_diff': np.mean(point_diffs),
        'expected_total_score': np.mean(total_scores),
        'median_point_diff': np.median(point_diffs),
        'point_diff_std': np.std(point_diffs),
        'total_score_std': np.std(total_scores),
        'simulation_data': {
            'home_scores': home_scores,
            'away_scores': away_scores,
            'point_diffs': point_diffs,
            'total_scores': total_scores
        }
    }
    
    return results

def visualize_simulation(simulation_results):
    """
    Creates visualizations of game simulation results
    
    Args:
        simulation_results: Dictionary with simulation results
    """
    if not simulation_results:
        print("No simulation results to visualize")
        return
    
    # Create figure
    plt.figure(figsize=(15, 10))
    
    # Extract data
    home_team = simulation_results['home_team']
    away_team = simulation_results['away_team']
    win_prob = simulation_results['simulated_win_probability']
    point_diffs = simulation_results['simulation_data']['point_diffs']
    total_scores = simulation_results['simulation_data']['total_scores']
    
    # Plot point differential histogram
    plt.subplot(2, 2, 1)
    sns.histplot(point_diffs, kde=True)
    plt.axvline(x=0, color='r', linestyle='--')
    plt.title(f'Point Differential: {home_team} vs {away_team}')
    plt.xlabel('Home Team Margin')
    plt.ylabel('Frequency')
    
    # Annotate win probability
    plt.annotate(f'Win Probability: {win_prob:.1%}',
                xy=(0.05, 0.95), xycoords='axes fraction',
                ha='left', va='top',
                bbox=dict(boxstyle='round', fc='white', alpha=0.8))
    
    # Plot total score histogram
    plt.subplot(2, 2, 2)
    sns.histplot(total_scores, kde=True)
    plt.title(f'Total Score: {home_team} vs {away_team}')
    plt.xlabel('Total Points')
    plt.ylabel('Frequency')
    
    # Calculate over/under distribution
    common_lines = [210, 215, 220, 225, 230, 235, 240]
    over_probs = [(np.array(total_scores) > line).mean() for line in common_lines]
    
    # Annotate over probabilities
    text = '\n'.join([f'O/U {line}: {prob:.1%} over' for line, prob in zip(common_lines, over_probs)])
    plt.annotate(text,
                xy=(0.95, 0.95), xycoords='axes fraction',
                ha='right', va='top',
                bbox=dict(boxstyle='round', fc='white', alpha=0.8))
    
    # Plot score scatterplot
    plt.subplot(2, 2, 3)
    home_scores = simulation_results['simulation_data']['home_scores']
    away_scores = simulation_results['simulation_data']['away_scores']
    
    # Create 2D histogram
    plt.hist2d(home_scores, away_scores, bins=20, cmap='Blues')
    plt.colorbar(label='Frequency')
    
    # Add diagonal line for equal scores
    min_score = min(min(home_scores), min(away_scores))
    max_score = max(max(home_scores), max(away_scores))
    plt.plot([min_score, max_score], [min_score, max_score], 'r--')
    
    plt.title(f'Score Distribution: {home_team} vs {away_team}')
    plt.xlabel(f'{home_team} Score')
    plt.ylabel(f'{away_team} Score')
    
    # Plot win probability by margin
    plt.subplot(2, 2, 4)
    
    # Calculate cumulative probability distribution
    margins = sorted(set([int(d) for d in point_diffs]))
    win_by_at_least = {}
    lose_by_at_least = {}
    
    for margin in range(max(margins) + 1):
        win_by_at_least[margin] = (np.array(point_diffs) >= margin).mean()
        
    for margin in range(abs(min(margins)) + 1):
        lose_by_at_least[margin] = (np.array(point_diffs) <= -margin).mean()
    
    # Plot
    plt.plot(list(win_by_at_least.keys()), list(win_by_at_least.values()), 'g-', label='Win by at least')
    plt.plot(list(lose_by_at_least.keys()), list(lose_by_at_least.values()), 'r-', label='Lose by at least')
    
    plt.title(f'Win/Loss Margin Probabilities: {home_team}')
    plt.xlabel('Margin (Points)')
    plt.ylabel('Probability')
    plt.legend()
    plt.grid(alpha=0.3)
    
    # Add common spread annotations
    common_spreads = [-7.5, -6.5, -3.5, -2.5, -1.5, 1.5, 2.5, 3.5, 6.5, 7.5]
    text = []
    
    for spread in common_spreads:
        if spread > 0:
            # Home team is underdog
            prob = (np.array(point_diffs) > -spread).mean()
            text.append(f'{home_team} +{spread}: {prob:.1%}')
        else:
            # Home team is favorite
            prob = (np.array(point_diffs) > abs(spread)).mean()
            text.append(f'{home_team} -{abs(spread)}: {prob:.1%}')
    
    plt.annotate('\n'.join(text[:5]),
                xy=(0.05, 0.95), xycoords='axes fraction',
                ha='left', va='top',
                bbox=dict(boxstyle='round', fc='white', alpha=0.8))
                
    plt.annotate('\n'.join(text[5:]),
                xy=(0.95, 0.95), xycoords='axes fraction',
                ha='right', va='top',
                bbox=dict(boxstyle='round', fc='white', alpha=0.8))
    
    plt.suptitle(f'Game Simulation: {home_team} vs {away_team}', fontsize=16)
    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.show()
    
    # Print summary statistics
    print(f"\nSimulation Results: {home_team} vs {away_team}")
    print(f"Win Probability: {win_prob:.1%}")
    print(f"Expected Score: {home_team} {simulation_results['expected_home_score']:.1f} - {away_team} {simulation_results['expected_away_score']:.1f}")
    print(f"Expected Point Differential: {simulation_results['expected_point_diff']:.1f} (±{simulation_results['point_diff_std']:.1f})")
    print(f"Expected Total Score: {simulation_results['expected_total_score']:.1f} (±{simulation_results['total_score_std']:.1f})")

# Simulate a matchup if we have team metrics
if not team_metrics.empty and 'models' in globals() and models:
    print("Running a sample game simulation...")
    
    # Get random teams from team metrics
    teams = team_metrics['team'].unique()
    
    if len(teams) >= 2:
        import random
        home_idx = random.randint(0, len(teams) - 1)
        away_idx = random.randint(0, len(teams) - 1)
        while away_idx == home_idx:  # Ensure teams are different
            away_idx = random.randint(0, len(teams) - 1)
            
        home_team = teams[home_idx]
        away_team = teams[away_idx]
        
        # Run simulation
        simulation_results = simulate_game(home_team, away_team, team_metrics, models)
        
        # Visualize results
        if simulation_results:
            visualize_simulation(simulation_results)
    else:
        print("Not enough teams available for simulation")

In [ ]:
# Cell 10 - Integration with Dynamic Recommendations

def generate_betting_recommendations(prediction):
    """
    Generates betting recommendations based on game predictions
    
    Args:
        prediction: Dictionary with game predictions
        
    Returns:
        Dictionary with betting recommendations
    """
    if not prediction:
        return {}
    
    # Extract key values
    home_team = prediction['home_team']
    away_team = prediction['away_team']
    home_score = prediction['predicted_home_score']
    away_score = prediction['predicted_away_score']
    win_prob = prediction['win_probability']
    point_diff = home_score - away_score
    total_score = home_score + away_score
    
    # Create recommendations
    recommendations = {}
    
    # Moneyline recommendation
    # Use Kelly criterion to determine value
    # Assumes fair odds based on win probability
    home_fair_odds = 1 / win_prob if win_prob > 0 else 1000
    away_fair_odds = 1 / (1 - win_prob) if win_prob < 1 else 1000
    
    # Translate to American odds for display
    if win_prob > 0.5:
        # Home team is favorite
        home_american = -100 / (home_fair_odds - 1) if home_fair_odds > 1 else -10000
        away_american = (away_fair_odds - 1) * 100
    else:
        # Away team is favorite
        home_american = (home_fair_odds - 1) * 100
        away_american = -100 / (away_fair_odds - 1) if away_fair_odds > 1 else -10000
    
    # Round to nearest 5
    home_american = round(home_american / 5) * 5
    away_american = round(away_american / 5) * 5
    
    recommendations["moneyline"] = {
        "home_fair_odds": round(home_fair_odds, 2),
        "away_fair_odds": round(away_fair_odds, 2),
        "home_american": int(home_american),
        "away_american": int(away_american),
        "recommendation": f"{home_team} moneyline" if win_prob > 0.55 else 
                         f"{away_team} moneyline" if win_prob < 0.45 else
                         "No strong moneyline value"
    }
    
    # Spread recommendation
    typical_spreads = [1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10.5, 11.5, 12.5, 13.5, 14.5]
    
    # Find closest typical spread
    closest_spread = min(typical_spreads, key=lambda x: abs(abs(point_diff) - x))
    
    # Determine favorite
    if point_diff > 0:
        # Home team is favorite
        spread_text = f"{home_team} -{closest_spread}" if point_diff > closest_spread else f"{away_team} +{closest_spread}"
    else:
        # Away team is favorite
        spread_text = f"{away_team} -{closest_spread}" if abs(point_diff) > closest_spread else f"{home_team} +{closest_spread}"
    
    recommendations["spread"] = {
        "projected_diff": round(point_diff, 1),
        "closest_typical_spread": closest_spread,
        "recommendation": spread_text
    }
    
    # Over/under recommendation
    typical_totals = [190.5, 195.5, 200.5, 205.5, 210.5, 215.5, 220.5, 225.5, 230.5, 235.5, 240.5, 245.5]
    
    # Find closest typical total
    closest_total = min(typical_totals, key=lambda x: abs(total_score - x))
    
    # Determine over/under recommendation (if projection is at least 4 points different)
    if total_score > closest_total + 4:
        ou_recommendation = f"OVER {closest_total}"
    elif total_score < closest_total - 4:
        ou_recommendation = f"UNDER {closest_total}"
    else:
        ou_recommendation = f"No strong over/under value at {closest_total}"
    
    recommendations["total"] = {
        "projected_total": round(total_score, 1),
        "closest_typical_total": closest_total,
        "recommendation": ou_recommendation
    }
    
    # Prop recommendations
    # This would be connected to player projections in a full implementation
    recommendations["props"] = {
        "home_team_props": f"Consider {home_team} team total OVER if set below {home_score - 3}" if home_score > 110 else
                          f"Consider {home_team} team total UNDER if set above {home_score + 3}" if home_score < 100 else
                          "No strong team total recommendation",
        "away_team_props": f"Consider {away_team} team total OVER if set below {away_score - 3}" if away_score > 110 else
                          f"Consider {away_team} team total UNDER if set above {away_score + 3}" if away_score < 100 else
                          "No strong team total recommendation"
    }
    
    return recommendations

def integrate_with_dynamic_recommendations(prediction):
    """
    Integrates pre-game predictions with the dynamic recommendations system
    
    Args:
        prediction: Dictionary with game predictions
        
    Returns:
        Dictionary with recommendations
    """
    try:
        # Try to import the dynamic recommendations module
        from models.dynamic_recommendation import generate_recommendations
        
        # Prepare model outputs for the recommendation engine
        model_outputs = {
            "win_probability": prediction['win_probability'],
            "momentum_shift": 0,  # No momentum in pre-game
            "projected_margin": prediction['predicted_home_score'] - prediction['predicted_away_score'],
            "total_projected_score": prediction['predicted_home_score'] + prediction['predicted_away_score'],
            "quarter": 0,  # Pre-game
            "time_remaining": 48  # Full game (48 minutes)
        }
        
        # Generate recommendations
        recommendations = generate_recommendations(model_outputs)
        return recommendations
        
    except ImportError:
        # Fallback to our built-in recommendation system if not available
        print("Dynamic recommendations module not available. Using built-in recommendations.")
        return generate_betting_recommendations(prediction)
    except Exception as e:
        print(f"Error integrating with dynamic recommendations: {e}")
        return generate_betting_recommendations(prediction)

# Demonstrate recommendations if we have predictions
if 'upcoming_predictions' in globals() and not upcoming_predictions.empty:
    # Get the first prediction
    first_prediction = upcoming_predictions.iloc[0].to_dict()
    
    # Generate recommendations
    recommendations = generate_betting_recommendations(first_prediction)
    
    # Display recommendations
    print("\nBetting Recommendations:")
    print(f"Game: {first_prediction['home_team']} vs {first_prediction['away_team']}")
    print(f"Predicted Score: {first_prediction['predicted_home_score']:.1f} - {first_prediction['predicted_away_score']:.1f}")
    print(f"Win Probability: {first_prediction['win_probability']:.1%}")
    
    print("\nMoneyline:")
    print(f"  {first_prediction['home_team']}: {recommendations['moneyline']['home_american']:+d}")
    print(f"  {first_prediction['away_team']}: {recommendations['moneyline']['away_american']:+d}")
    print(f"  Recommendation: {recommendations['moneyline']['recommendation']}")
    
    print("\nSpread:")
    print(f"  Projected Point Differential: {recommendations['spread']['projected_diff']:+.1f}")
    print(f"  Recommendation: {recommendations['spread']['recommendation']}")
    
    print("\nTotal:")
    print(f"  Projected Total: {recommendations['total']['projected_total']:.1f}")
    print(f"  Recommendation: {recommendations['total']['recommendation']}")
    
    print("\nProps:")
    print(f"  {first_prediction['home_team']}: {recommendations['props']['home_team_props']}")
    print(f"  {first_prediction['away_team']}: {recommendations['props']['away_team_props']}")

In [ ]:
# Cell 11 - API Integration for Web Application

def create_api_response(prediction):
    """
    Formats prediction data for API response
    
    Args:
        prediction: Dictionary with game predictions
        
    Returns:
        Dictionary formatted for API
    """
    if not prediction:
        return {}
    
    # Generate recommendations
    try:
        from models.dynamic_recommendation import generate_recommendations
        
        # Prepare model outputs for the recommendation engine
        model_outputs = {
            "win_probability": prediction['win_probability'],
            "momentum_shift": 0,  # No momentum in pre-game
            "projected_margin": prediction['predicted_home_score'] - prediction['predicted_away_score'],
            "total_projected_score": prediction['predicted_home_score'] + prediction['predicted_away_score'],
            "quarter": 0,  # Pre-game
            "time_remaining": 48  # Full game (48 minutes)
        }
        
        # Generate recommendations
        recommendations = generate_recommendations(model_outputs)
    except:
        # Fallback to built-in recommendations
        recommendations = generate_betting_recommendations(prediction)
    
    # Format for API
    api_response = {
        "game": {
            "id": prediction['game_id'],
            "date": prediction['game_date'].strftime('%Y-%m-%d') if isinstance(prediction['game_date'], datetime) else prediction['game_date'],
            "home_team": prediction['home_team'],
            "away_team": prediction['away_team'],
            "status": "upcoming",
            "start_time": "TBD"  # Would be populated from schedule in real implementation
        },
        "prediction": {
            "home_score": round(prediction['predicted_home_score'], 1),
            "away_score": round(prediction['predicted_away_score'], 1),
            "point_differential": round(prediction['predicted_home_score'] - prediction['predicted_away_score'], 1),
            "total_score": round(prediction['predicted_home_score'] + prediction['predicted_away_score'], 1),
            "win_probability": round(prediction['win_probability'], 3),
            "timestamp": datetime.now().isoformat(),
            "model_version": "1.0"
        },
        "recommendations": recommendations,
        "metrics": {
            "home_rest_days": prediction.get('home_rest', 0),
            "away_rest_days": prediction.get('away_rest', 0),
            "home_back_to_back": prediction.get('home_back_to_back', False),
            "away_back_to_back": prediction.get('away_back_to_back', False)
        }
    }
    
    return api_response

def simulate_api_response():
    """Simulates API responses for future integration with web app"""
    
    # Check if we have upcoming predictions
    if 'upcoming_predictions' in globals() and not upcoming_predictions.empty:
        predictions = upcoming_predictions
    else:
        # Try to generate new predictions
        if 'prediction_models' in globals() and prediction_models:
            predictions = predict_all_upcoming_games(prediction_models)
        else:
            print("No predictions or models available")
            return {}
    
    if predictions.empty:
        print("No predictions available")
        return {}
    
    # Format all predictions for API
    api_responses = []
    
    for _, prediction in predictions.iterrows():
        api_response = create_api_response(prediction.to_dict())
        api_responses.append(api_response)
    
    print(f"Created API responses for {len(api_responses)} games")
    
    # Display sample
    if api_responses:
        import json
        print("\nSample API Response for first game:")
        print(json.dumps(api_responses[0], indent=2, default=str))
    
    return api_responses

# Simulate API responses
api_responses = simulate_api_response()

In [ ]:
# Cell 12 - Schedule Model Training and Predictions

from apscheduler.schedulers.background import BackgroundScheduler
import pytz
import time
import os

def scheduled_model_training():
    """Scheduled job for model training"""
    print(f"\n=== Scheduled Model Training ({datetime.now().isoformat()}) ===")
    
    # Load historical games
    historical_df = load_historical_games(days_lookback=180)
    
    if historical_df.empty:
        print("No historical data available. Skipping training.")
        return
    
    # Calculate team metrics
    metrics_df = calculate_team_metrics(historical_df)
    
    if metrics_df.empty:
        print("No team metrics available. Skipping training.")
        return
    
    # Build features
    features_df = build_pregame_features(historical_df, metrics_df, lookback_days=120)
    
    if features_df.empty:
        print("No features available. Skipping training.")
        return
    
    # Train models
    models = train_multiple_models(features_df)
    
    # Save global reference to models
    global prediction_models
    prediction_models = models
    
    print("Model training completed successfully.")

def scheduled_predictions():
    """Scheduled job for generating predictions"""
    print(f"\n=== Scheduled Predictions ({datetime.now().isoformat()}) ===")
    
    # Check if we have models
    if 'prediction_models' not in globals() or not prediction_models:
        print("No models available. Attempting to load saved models...")
        
        # Try to load saved models
        saved_models = {}
        for target in ['home_score', 'away_score', 'point_diff', 'total_score']:
            model_path = MODELS_DIR / f"pregame_{target}_model.pkl"
            if model_path.exists():
                model = joblib.load(model_path)
                saved_models[target] = {'model': model, 'metrics': {}}
        
        if not saved_models:
            print("No saved models found. Skipping predictions.")
            return
        
        # Use loaded models
        global prediction_models
        prediction_models = saved_models
    
    # Generate predictions
    predictions = predict_all_upcoming_games(prediction_models)
    
    if predictions.empty:
        print("No predictions generated.")
        return
    
    # Save predictions to file
    predictions_file = MODELS_DIR / f"pregame_predictions_{datetime.now().strftime('%Y%m%d')}.csv"
    predictions.to_csv(predictions_file, index=False)
    print(f"Saved {len(predictions)} predictions to {predictions_file}")
    
    # Update global reference
    global upcoming_predictions
    upcoming_predictions = predictions
    
    # Generate API responses
    api_responses = []
    for _, prediction in predictions.iterrows():
        api_response = create_api_response(prediction.to_dict())
        api_responses.append(api_response)
    
    # Save API responses to file
    import json
    api_file = MODELS_DIR / f"api_responses_{datetime.now().strftime('%Y%m%d')}.json"
    with open(api_file, 'w') as f:
        json.dump(api_responses, f, default=str)
    
    print(f"Saved API responses to {api_file}")
    
    # Print summary
    print("\nPrediction Summary:")
    for _, pred in predictions.iterrows():
        print(f"{pred['home_team']} vs {pred['away_team']}: " +
              f"{pred['predicted_home_score']:.1f}-{pred['predicted_away_score']:.1f} " +
              f"(Win prob: {pred['win_probability']:.1%})")

def run_scheduler_demo(run_time=30):
    """
    Runs a demonstration of the scheduling system
    
    Args:
        run_time: Seconds to run the demo scheduler
    """
    print(f"Starting scheduler demo (will run for {run_time} seconds)...")
    
    scheduler = BackgroundScheduler(timezone=pytz.timezone('America/Los_Angeles'))
    
    # Add jobs
    scheduler.add_job(
        scheduled_model_training,
        'interval',
        minutes=15,  # Every 15 minutes in demo (would be daily in production)
        id='model_training_job'
    )
    
    scheduler.add_job(
        scheduled_predictions,
        'interval',
        minutes=5,  # Every 5 minutes in demo (would be hourly in production)
        id='predictions_job'
    )
    
    # Start the scheduler
    scheduler.start()
    print("Scheduler started. Running initial jobs...")
    
    # Run initial jobs
    scheduled_model_training()
    scheduled_predictions()
    
    # Keep the script running
    try:
        print(f"Scheduler running... (press Ctrl+C to exit after {run_time} seconds)")
        time.sleep(run_time)
    except (KeyboardInterrupt, SystemExit):
        pass
    finally:
        scheduler.shutdown()
        print("Scheduler shutdown.")

# Ask if user wants to run the scheduler demo
print("\nDo you want to run the scheduler demo? (y/n)")
run_demo = input().lower().strip() == 'y'

if run_demo:
    run_scheduler_demo(run_time=30)
else:
    print("Scheduler demo skipped. You can run it later by calling run_scheduler_demo().")

In [ ]:
# Cell 13 - Save and Export Notebook State

import json
import os
from datetime import datetime

def save_notebook_state():
    """Saves key variables and model state for later use"""
    
    # Create state directory if it doesn't exist
    state_dir = MODELS_DIR / "state"
    state_dir.mkdir(exist_ok=True)
    
    # Save state timestamp
    state_time = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # List of variables to save metadata for
    save_vars = [
        'historical_games', 'team_metrics', 'team_stats',
        'pregame_features', 'upcoming_predictions'
    ]
    
    metadata = {
        'timestamp': state_time,
        'saved_variables': {},
        'saved_models': []
    }
    
    # Save variable metadata
    for var_name in save_vars:
        if var_name in globals():
            var = globals()[var_name]
            if isinstance(var, pd.DataFrame):
                # Save DataFrame info
                metadata['saved_variables'][var_name] = {
                    'type': 'DataFrame',
                    'shape': var.shape,
                    'columns': list(var.columns)
                }
                
                # Save DataFrame to CSV
                csv_path = state_dir / f"{var_name}_{state_time}.csv"
                var.to_csv(csv_path, index=False)
                metadata['saved_variables'][var_name]['path'] = str(csv_path)
    
    # Save models
    for model_type in ['home_score', 'away_score', 'point_diff', 'total_score']:
        model_path = MODELS_DIR / f"pregame_{model_type}_model.pkl"
        if model_path.exists():
            # Record model in metadata
            metadata['saved_models'].append({
                'type': model_type,
                'path': str(model_path)
            })
    
    # Save metadata
    metadata_path = state_dir / f"notebook_state_{state_time}.json"
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2, default=str)
    
    print(f"Notebook state saved to {metadata_path}")
    return metadata

def export_for_web_app():
    """Exports models and data for use in web application"""
    
    # Create export directory
    export_dir = pathlib.Path(os.getcwd()) / "exports"
    export_dir.mkdir(exist_ok=True)
    
    export_time = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Export models
    models_exported = 0
    for model_type in ['home_score', 'away_score', 'point_diff', 'total_score']:
        model_path = MODELS_DIR / f"pregame_{model_type}_model.pkl"
        if model_path.exists():
            export_path = export_dir / f"pregame_{model_type}_model.pkl"
            import shutil
            shutil.copy(model_path, export_path)
            models_exported += 1
    
    # Export team metrics if available
    metrics_exported = False
    if 'team_metrics' in globals() and not team_metrics.empty:
        metrics_path = export_dir / "team_metrics.csv"
        team_metrics.to_csv(metrics_path, index=False)
        metrics_exported = True
    
    # Export latest predictions if available
    predictions_exported = False
    if 'upcoming_predictions' in globals() and not upcoming_predictions.empty:
        predictions_path = export_dir / "latest_predictions.csv"
        upcoming_predictions.to_csv(predictions_path, index=False)
        predictions_exported = True
    
    # Create manifest
    manifest = {
        'export_time': export_time,
        'models_exported': models_exported,
        'team_metrics_exported': metrics_exported,
        'predictions_exported': predictions_exported,
        'files': [f.name for f in export_dir.glob('*')]
    }
    
    manifest_path = export_dir / "manifest.json"
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)
    
    print(f"Exported {models_exported} models and data to {export_dir}")
    return manifest

# Save current notebook state
notebook_state = save_notebook_state()

# Export for web app
export_manifest = export_for_web_app()

print("\nNotebook execution complete!")
print("Pre-game prediction models and data have been trained and exported.")